# 03 — silver_customers
**Entrée :** `bronze.customers` — **Sortie :** `silver.silver_customers` (99 441 lignes)
**Nettoyage :** ville en minuscules sans espaces, État en majuscules, code postal **gardé en texte** (zéros initiaux : `09790`).
**Rappel :** `customer_id` = un par commande ; `customer_unique_id` = la vraie personne (96 096).

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
# Les tables Bronze sont supposées s'appeler : orders, customers, order_items, order_payments,
# order_reviews, products, sellers, product_category_name_translation.
# Si les tiennes ont d'autres noms, change-les dans les requêtes (Ctrl+F).

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    """Exécute une requête qui renvoie une seule valeur."""
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    """Compare ce qu'on obtient à ce qu'on attend (valeurs de ton document de travail)."""
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_customers AS
SELECT
    customer_id,
    customer_unique_id,
    CAST(customer_zip_code_prefix AS STRING) AS customer_zip_code_prefix,
    LOWER(TRIM(customer_city))  AS customer_city,
    UPPER(TRIM(customer_state)) AS customer_state,
    _loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM {BRONZE}.customers
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
t = f"{SILVER}.silver_customers"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 99441)
check("customer_id unique (doublons)", scalar(f"SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM {t}"), 0)
check("customer_id NULL", scalar(f"SELECT COUNT(*) FROM {t} WHERE customer_id IS NULL"), 0)
check("personnes distinctes", scalar(f"SELECT COUNT(DISTINCT customer_unique_id) FROM {t}"), 96096)
check("villes distinctes", scalar(f"SELECT COUNT(DISTINCT customer_city) FROM {t}"), 4119)
check("codes postaux de longueur != 5", scalar(f"SELECT COUNT(*) FROM {t} WHERE LENGTH(customer_zip_code_prefix) <> 5"), 0)

✅ OK     lignes : obtenu = 99441 | attendu = 99441
✅ OK     customer_id unique (doublons) : obtenu = 0 | attendu = 0
✅ OK     customer_id NULL : obtenu = 0 | attendu = 0
✅ OK     personnes distinctes : obtenu = 96096 | attendu = 96096
✅ OK     villes distinctes : obtenu = 4119 | attendu = 4119
✅ OK     codes postaux de longueur != 5 : obtenu = 0 | attendu = 0


## À regarder (question R13, à documenter)
Les villes contiennent-elles des accents ? On ne les enlève pas ici : on regarde juste l'effet.

In [0]:
try:
    spark.sql(f"""
        SELECT COUNT(DISTINCT customer_city) AS villes_avec_accents,
               COUNT(DISTINCT translate(customer_city,
                   'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc')) AS villes_sans_accents
        FROM {SILVER}.silver_customers
    """).show()
except Exception as e:
    print("Cellule facultative ignorée :", str(e)[:200])

+-------------------+-------------------+
|villes_avec_accents|villes_sans_accents|
+-------------------+-------------------+
|               4119|               4119|
+-------------------+-------------------+



## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?